# Silver Layer — ERP Customer Location (LOC_A101)
Clean and normalize `erp_loc_a101`.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "silver",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## Read Bronze Table

In [ ]:
df = session.table(f"bronze.erp_loc_a101")

## Silver Transformations

### Trimming

In [ ]:
from clickzetta.zettapark.types import StringType
from clickzetta.zettapark import functions as F

for field in df.schema.fields:
    if isinstance(field.datatype, StringType):
        df = df.with_column(field.name, F.trim(F.col(field.name)))

### Customer ID Cleanup
Remove hyphens from customer ID.

In [ ]:
df = df.with_column("cid", F.regexp_replace(F.col("cid"), F.lit("-"), F.lit("")))

### Country Normalization

In [ ]:
df = df.with_column(
    "cntry",
    F.when(F.col("cntry") == "DE", "Germany")
     .when(F.col("cntry").isin("US", "USA"), "United States")
     .when(F.col("cntry").is_null() | (F.col("cntry") == ""), "n/a")
     .otherwise(F.col("cntry"))
)

### Rename Columns

In [ ]:
RENAME_MAP = {
    "cid":   "customer_number",
    "cntry": "country",
}
for old, new in RENAME_MAP.items():
    df = df.with_column_renamed(old, new)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Silver Table

In [ ]:
df.write.save_as_table(f"silver.erp_customer_location", mode="overwrite")
print("erp_customer_location OK")

## Verify

In [ ]:
session.table(f"silver.erp_customer_location").limit(5).show()